# **Human-in-the-Loop (interrupt)**

Sometimes you want to **pause** a graph mid-run and ask a human to approve or reject before continuing.
`interrupt()` pauses the graph and stores a question/payload in state.
The human resumes the graph by calling `graph.invoke(Command(resume=value), config)`.

In [ ]:
from typing import Literal, Optional, TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt   # interrupt pauses execution; Command resumes it

# ── State ─────────────────────────────────────────────────────────────────────
class ApprovalState(TypedDict):
    action_details: str                                        # what action needs approval
    status: Optional[Literal["pending", "approved", "rejected"]]  # outcome after human decision

### **Nodes**

In [ ]:
def approval_node(state: ApprovalState) -> Command[Literal["proceed", "cancel"]]:
    """
    Pauses the graph and waits for a human decision.
    interrupt() suspends execution and stores the payload (the question + details).
    When the graph is resumed with Command(resume=value), `decision` receives that value.
    """
    decision = interrupt({
        "question": "Approve this action?",
        "details":  state["action_details"],
    })

    # Route to the correct next node based on the human's decision (True/False)
    return Command(goto="proceed" if decision else "cancel")


def proceed_node(state: ApprovalState):
    """Runs when the human approved — marks the action as approved."""
    return {"status": "approved"}


def cancel_node(state: ApprovalState):
    """Runs when the human rejected — marks the action as rejected."""
    return {"status": "rejected"}

### **Build and Compile the Graph**
Checkpointing is **required** for Human-in-the-Loop — the graph state must be persisted between the pause and the resume.

In [ ]:
builder = StateGraph(ApprovalState)
builder.add_node("approval", approval_node)
builder.add_node("proceed",  proceed_node)
builder.add_node("cancel",   cancel_node)

builder.add_edge(START,     "approval")   # start by asking for approval
builder.add_edge("proceed", END)           # approval_node routes here if approved
builder.add_edge("cancel",  END)           # approval_node routes here if rejected

# MemorySaver is required: the graph must persist state between invoke() calls
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())

### **First invoke — graph pauses at interrupt()**

In [ ]:
config = {"configurable": {"thread_id": "approval-456"}}

# invoke() runs until it hits interrupt() inside approval_node, then pauses
initial = graph.invoke(
    {"action_details": "Send Daily Summary Email", "status": "pending"},
    config=config,
)

# The graph is now suspended. __interrupt__ shows the question the human needs to answer.
print(initial["__interrupt__"])   # [Interrupt(value={'question': ..., 'details': ...})]

### **Resume — human provides their decision**

In [ ]:
# Command(resume=True)  → human approved   → proceed_node runs
# Command(resume=False) → human rejected  → cancel_node runs

# Example: reject the action
resumed = graph.invoke(Command(resume=False), config=config)
print(resumed["status"])   # "rejected"

# Try with True to see "approved":
# resumed = graph.invoke(Command(resume=True), config=config)
# print(resumed["status"])  # "approved"